In [1]:
import numpy as np
import tensorflow as tf
from keras.datasets import cifar10
from keras.utils import np_utils
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from art.attacks.evasion import SaliencyMapMethod
from art.estimators.classification  import KerasClassifier
from sklearn.metrics import accuracy_score

In [2]:
tf.compat.v1.disable_eager_execution()

In [3]:
import tensorflow as tf
import json
# download mnist data and split into train and test sets
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
# reshape data to fit model
X_train, X_test = X_train/255, X_test/255
# one-hot encode target column
y_train = tf.keras.utils.to_categorical(y_train)
y_test = tf.keras.utils.to_categorical(y_test)
# create model
model = Sequential()
model.add(Conv2D(32, (3, 3), padding='same', input_shape=(32, 32, 3), activation='relu'))
model.add(Conv2D(32, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(10, activation='softmax'))

In [4]:
# compile model using accuracy as a measure of model performance
model.compile(optimizer='adam', loss='categorical_crossentropy',
              metrics=['accuracy'])

# train model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5)

json.dump({'model': model.to_json()}, open("model.json", "w"))
model.save_weights("model_weights.h5")


Train on 50000 samples, validate on 10000 samples
Epoch 1/5
50000/50000 [==============================] - ETA: 0s - loss: 1.4927 - accuracy: 0.4544

C:\Users\nidhi.jain1\Anaconda3\lib\site-packages\keras\engine\training_v1.py:2333: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates = self.state_updates


50000/50000 [==============================] - 359s 7ms/sample - loss: 1.4927 - accuracy: 0.4544 - val_loss: 1.1030 - val_accuracy: 0.6060
Epoch 2/5
50000/50000 [==============================] - 515s 10ms/sample - loss: 1.0959 - accuracy: 0.6132 - val_loss: 0.9223 - val_accuracy: 0.6775
Epoch 3/5
50000/50000 [==============================] - 387s 8ms/sample - loss: 0.9517 - accuracy: 0.6658 - val_loss: 0.8446 - val_accuracy: 0.7107
Epoch 4/5
50000/50000 [==============================] - 360s 7ms/sample - loss: 0.8546 - accuracy: 0.7051 - val_loss: 0.8046 - val_accuracy: 0.7192
Epoch 5/5
50000/50000 [==============================] - 249s 5ms/sample - loss: 0.7924 - accuracy: 0.7212 - val_loss: 0.7676 - val_accuracy: 0.7362


In [5]:
#Create a KerasClassifier for the model
classifier = KerasClassifier(model=model, clip_values=(0, 1),  use_logits=False)
# Generate adversarial examples
x_test = X_test # your test data
y = y_test # the true labels for the test data
jsma = SaliencyMapMethod(classifier=classifier, theta=0.3, gamma=0.1)
#x_test_adv = jsma.generate(x=x_test, y=y)
x_test_adv = jsma.generate(x_test)

C:\Users\nidhi.jain1\Anaconda3\lib\site-packages\keras\engine\training_v1.py:2357: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


JSMA:   0%|          | 0/10000 [00:00<?, ?it/s]

In [6]:
# Evaluate the model's accuracy before adv attack on the test dataset
score1 = model.evaluate(x_test, y_test, verbose=0)
print('Test accuracy:', score1[1])

# Evaluate the model's accuracy on the test dataset
model.fit(X_train, y_train, batch_size=32, epochs=5, validation_data=(x_test_adv, y_test))
score1 = model.evaluate(x_test_adv, y_test, verbose=0)
print('Test accuracy:', score1[1])

x_test_adv
x_test
print(f"X_train shape: {x_test_adv.shape}")
print(f"y_train shape: {x_test.shape}")

Test accuracy: 0.7362
Train on 50000 samples, validate on 10000 samples
Epoch 1/5
50000/50000 [==============================] - 148s 3ms/sample - loss: 0.7450 - accuracy: 0.7395 - val_loss: 1.3182 - val_accuracy: 0.5227
Epoch 2/5
50000/50000 [==============================] - 171s 3ms/sample - loss: 0.7046 - accuracy: 0.7531 - val_loss: 1.2700 - val_accuracy: 0.5360
Epoch 3/5
50000/50000 [==============================] - 174s 3ms/sample - loss: 0.6695 - accuracy: 0.7664 - val_loss: 1.3747 - val_accuracy: 0.5191
Epoch 4/5
50000/50000 [==============================] - 180s 4ms/sample - loss: 0.6483 - accuracy: 0.7733 - val_loss: 1.2594 - val_accuracy: 0.5502
Epoch 5/5
50000/50000 [==============================] - 179s 4ms/sample - loss: 0.6227 - accuracy: 0.7817 - val_loss: 1.1909 - val_accuracy: 0.5796
Test accuracy: 0.5796
X_train shape: (10000, 32, 32, 3)
y_train shape: (10000, 32, 32, 3)
